# Exploratory Data Analysis of the HMDA Dataset (2007 - 2017)

In this notebook, I analyse basic trends to get a better understanding of the HMDA dataset.

In [1]:
# Libraries

import os
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import numpy as np

from matplotlib.ticker import StrMethodFormatter

In [2]:
os.getcwd()
os.chdir("../data/processed")

## 1. Categorical variable exploration

The variables are: income_000s, loan_amount_000s, tract_to_msamd_income

### A. Income

In [ ]:
# Import data set
hmda = pd.read_csv("cleaned_missing.csv")

In [ ]:
# There are clearly outliers in the dataset
max_income = hmda['income_000s'].max() * 1000
print("The max income is", f'{max_income: ,}') # 60,789,000 is the max income!

min_income = hmda['income_000s'].min() * 1000
print("The min income is", f'{min_income: ,}') # 1,000 is the min income, which makes sense.

# Define a reasonable range to view
ax = sns.histplot(data=hmda, x="income_000s", bins=500)
ax.set(xlabel = "Income (000s)", ylabel = "Count")
ax.set_xlim(0, 2000)

In [ ]:
# Apply log transformation to the loan data
income_df = hmda[['income_000s']]
log_income = np.log(income_df)

# Plot
ax = sns.histplot(data = log_income, x = "income_000s", bins = 500)
ax.set(xlabel = "Log Income (000s)", ylabel = "Count")

Given that there are clear outliers on the right end, I set the maximum plausible income to log_income_000s = 8.5, which corresponds to around $5 million (per year). While there may genuinely be people earning more than this amount per year, this is a reasonable number that allows for extremely high incomes while not allowing nonsensical outliers. There are no clear outliers on the left side (as minimum income is still reasonable at $1000), so I only drop values where income is greater than $5 million.

In [ ]:
# Calculate instances of outlier rows
count_outlier_income = (hmda['income_000s'] > 5000).sum()
pct_outlier_income = round(count_outlier_income/len(hmda) * 100, 2) 

print("The number of outlier rows is", f'{count_outlier_income: ,}', ", which is", f'{pct_outlier_income}', "%.")

# Drop instances of outlier rows
hmda = hmda.loc[hmda['income_000s'] <= 5000]

### B. Loan amount

In [ ]:
# There are clearly outliers in the dataset
max_loan = hmda['loan_amount_000s'].max() * 1000
print("The max loan is", f'{max_loan: ,}') # 475 million is the max loan!

min_loan = hmda['loan_amount_000s'].min() * 1000
print("The min loan is", f'{min_loan: ,}') # 1,000 is the min loan, which makes sense.

# Define a reasonable range to view
ax = sns.histplot(data=hmda, x="loan_amount_000s", bins=5000)
ax.set(xlabel = "Loan (000s)", ylabel = "Count")
ax.set_xlim(0, 2000)

In [ ]:
# Apply log transformation to the loan data
loan_df = hmda[['loan_amount_000s']]
log_loan = np.log(loan_df)

# Plot
ax = sns.histplot(data = log_loan, x = "loan_amount_000s", bins = 500)
ax.set(xlabel = "Log Loan (000s)", ylabel = "Count")

In [ ]:
# Examine the spike
# Extract the data
x = log_loan["loan_amount_000s"]

# Compute histogram with same number of bins as seaborn
counts, bin_edges = np.histogram(x, bins=500)

# Find the tallest bin
max_idx = np.argmax(counts)

# Bin information
x_left = bin_edges[max_idx]
x_right = bin_edges[max_idx + 1]
x_center = (x_left + x_right) / 2
y_value = counts[max_idx]

print(f"Peak bin:")
print(f"Center = {x_center}")
print(f"Count = {y_value}")

The peak at around 6.03 corresponds to around $417,000. This amount is the "conforming loan limit", i.e. the maximum size loan Fannie Mae and Freddie Mac would purchase, as it is easy to sell off. Loan requests just above that limit are shifted down to the conforming loan limit to get better interest rates. This is not an error of the data, but rather a facet of the loan market. This is why, just to the right of the peak, there is a sharp drop before increasing again.

Once again, there are clear outliers and I set all maximum loans to $5 million, which maintains a very high loan amount while getting rid of outliers.

In [ ]:
# Calculate instances of outlier rows
count_outlier_loan = (hmda['loan_amount_000s'] > 5000).sum()
pct_outlier_loan = round(count_outlier_loan/len(hmda) * 100, 2) 

print("The number of outlier rows is", f'{count_outlier_loan: ,}', ", which is", f'{pct_outlier_loan}', "%.")

# Drop instances of outlier rows
hmda = hmda.loc[hmda['loan_amount_000s'] <= 5000]

### C. Tract to MSA/MD income

In [ ]:
# Check for signs of outliers
max_ratio = round(hmda['tract_to_msamd_income'].max(),2)
print("The max ratio is", f'{max_ratio: ,}') # 475 million is the max loan!

min_ratio = hmda['tract_to_msamd_income'].min()
print("The min ratio is", f'{min_ratio: ,}') # 475 million is the max loan!

# Define a reasonable range to view
plt.clf()
ax = sns.histplot(data = hmda, x = "tract_to_msamd_income", bins=50)
ax.set(xlabel = "Tract to MSA/MD Income", ylabel = "Count")
ax.set_xlim(0, 500)

The distribution is centered around 100 (i.e. 100%), which is what we would expect given that the variable tells us the ratio of the person's income compared to the average in the county.

There are instances where the tract to MSA/MD income is 0, suggesting that the person has no income when applying for the loan. This is likely not an outlier.

To calculate whether there are outliers on the right-hand-side of the histogram, I use Tukey's method, which says that all values outside the interval of [Lower quartile - 1.5 * IQR, Upper quartile + 1.5 * IQR] (where IQR is the inter-quartile range) are outliers.

In [ ]:
quartiles = hmda['tract_to_msamd_income'].quantile([0.25, 0.75])
iqr = quartiles[0.75] - quartiles[0.25]

lower_bound = quartiles[0.25] - 1.5*iqr
upper_bound = quartiles[0.75] + 1.5*iqr

print("The lower bound is", round(lower_bound, 2))
print("The upper bound is", round(upper_bound, 2))

The upper bound still cuts off a significant tail, and incomes of 450 times the average is not implausible in California. I do not drop outliers, but I still log-transform the data for consistency.

In [ ]:
# Apply log transformation to the tract to MSA/MD income data
ratio_df = hmda[['tract_to_msamd_income']]
log_ratio = np.log(ratio_df)

# Plot
ax = sns.histplot(data = log_ratio, x = "tract_to_msamd_income", bins = 500)
ax.set(xlabel = "Log Tract to MSA/MD Income", ylabel = "Count")
ax.set_xlim(0, 10)

### D. Final log-transformed categorical variables

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 5))

sns.histplot(data = log_income, x = "income_000s", bins = 500, ax = axes[0])
axes[0].set(xlabel = "Log Applicant Income (000s)", ylabel = "Count")
axes[0].set_xlim(2, 9)
axes[0].set_title("Log-Transformed Applicant Income (000s)")

sns.histplot(data = log_loan, x = "loan_amount_000s", bins = 500, ax = axes[1])
axes[1].set(xlabel = "Loan Applicant Amount (000s)", ylabel = "Count")
axes[1].set_xlim(2, 9)
axes[1].set_title("Log-Transformed Loan Amount (000s)")

sns.histplot(data = log_ratio, x = "tract_to_msamd_income", bins = 500, ax = axes[2])
axes[2].set(xlabel = "Log Tract to MSA/MD Income", ylabel = "Count")
axes[2].set_xlim(2, 6)
axes[2].set_title("Log-Transformed Tract to MSA/MD Income")

# Add comma separators to all y-axes
for ax in axes:
    ax.yaxis.set_major_formatter(StrMethodFormatter('{x:,.0f}'))

plt.tight_layout()
plt.show()

I do not repeat the exercise for each year, as (1) outlier decisions are generously high, and there will be no years that have outliers past this number; (2) the central limit theorem reassures me that for each year the data will still look roughly log-normal; and (3) year-by-year exploration of the dataset is more suited to analysis stage, when "real-time" data begin arriving.

### E. Examining the "conforming loan spike" by year

In [ ]:
# Create a loop that shows the spikes by year

years = range(2007, 2018)

fig, axes = plt.subplots(3, 4, figsize=(16, 10))
axes = axes.flatten()

for i, year in enumerate(years):

    x = hmda.loc[hmda['year'] == year]
    log_loan = np.log(x['loan_amount_000s'])

    sns.histplot(
        x=log_loan,
        bins=200,
        ax=axes[i]
    )

    axes[i].set_title(str(year))
    axes[i].set_xlim(2, 9)

# Remove the unused 12th subplot
fig.delaxes(axes[11])

plt.tight_layout()
plt.show()

In [ ]:
# Create a dataframe that shows the value of the spikes

results = []

for year in range(2007, 2018):

    # Filter to year
    x = hmda.loc[hmda['year'] == year, 'loan_amount_000s']

    # Log transform
    log_loan = np.log(x)

    # Histogram
    counts, bin_edges = np.histogram(log_loan, bins=500)

    # Peak bin
    max_idx = np.argmax(counts)

    x_left = bin_edges[max_idx]
    x_right = bin_edges[max_idx + 1]
    x_center = (x_left + x_right) / 2

    results.append({
        'Year': year,
        'Peak log loan': round(x_left, 2),
    })

# Display results

peak_df = pd.DataFrame(results)

# Convert to real (not log) values
peak_df["Peak loan (USD)"] = round(np.exp(peak_df["Peak log loan"]) * 1000, 2)

# Comparison with real conforming loan limits
conforming_loan_limits = [417000] * 10
conforming_loan_limits.append(424000)
peak_df["Conforming loan limits (USD)"] = conforming_loan_limits

peak_df.style.format({
    "Peak loan (USD)": "{:,.2f}",
    "Conforming loan limits (USD)": "{:,.2f}"
})

This matches the pattern given by the US Government (https://www.fhfa.gov/blog/insights/the-dynamics-of-fhfa-conforming-loan-limits-and-house-prices). Any differences are due to binning amounts.

## 1. Continuous variable exploration

The variables are: action_taken, race, sex, ethnicity, loan_type, lien_status